In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2019 NVIDIA Corporation
Built on Sun_Jul_28_19:12:52_Pacific_Daylight_Time_2019
Cuda compilation tools, release 10.1, V10.1.243


In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.autograd import Variable
from torchvision import datasets, transforms
from torchsummary import summary

In [2]:
CUDA = torch.cuda.is_available()
device = torch.device("cuda" if CUDA else "cpu")
print(device)

cuda


In [3]:
class inceptionv1_block(nn.Module):
    def __init__(self, in_channels, out_channels1, out_channels2_step1, out_channels2_step2, out_channels3_step1, out_channels3_step2, out_channels4):
        super(inceptionv1_block, self).__init__()
        self.branch1_conv = nn.Sequential(nn.Conv2d(in_channels=in_channels, out_channels=out_channels1, kernel_size=1),
                          nn.ReLU(inplace=True))
        
        self.branch2_conv1 = nn.Sequential(nn.Conv2d(in_channels=in_channels, out_channels=out_channels2_step1, kernel_size=1),
                          nn.ReLU(inplace=True))
        self.branch2_conv2 = nn.Sequential(nn.Conv2d(in_channels=out_channels2_step1, out_channels=out_channels2_step2, kernel_size=3, padding=1),
                          nn.ReLU(inplace=True))
        
        self.branch3_conv1 = nn.Sequential(nn.Conv2d(in_channels=in_channels, out_channels=out_channels3_step1, kernel_size=1),
                          nn.ReLU(inplace=True))
        self.branch3_conv2 = nn.Sequential(nn.Conv2d(in_channels=out_channels3_step1, out_channels=out_channels3_step2, kernel_size=5, padding=2),
                          nn.ReLU(inplace=True))
        
        self.branch4_maxpooling = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.branch4_conv1 = nn.Sequential(nn.Conv2d(in_channels=in_channels, out_channels=out_channels4, kernel_size=1),
                          nn.ReLU(inplace=True))
     
    def forward(self, x):
        out1 = self.branch1_conv(x)
        out2 = self.branch2_conv2(self.branch2_conv1(x))
        out3 = self.branch3_conv2(self.branch3_conv1(x))
        out4 = self.branch4_conv1(self.branch4_maxpooling(x))
        out = torch.cat([out1, out2, out3, out4], dim=1)

        return out    

In [4]:
class auxiliary_classifiers(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(auxiliary_classifiers, self).__init__()
        self.avgpooling = nn.AvgPool2d(kernel_size=5, stride=3)
        
        self.conv = nn.Conv2d(in_channels=in_channels, out_channels=128, kernel_size=1)
        
        self.fc1 = nn.Linear(in_features=128*4*4, out_features=1024)

        self.fc2 = nn.Linear(in_features=1024, out_features=out_channels)
     
    def forward(self, x):
        x = self.avgpooling(x)
        x = F.relu(self.conv(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=0.5)
        x = self.fc2(x)

        return x   

In [5]:
class InceptionV101(nn.Module):
    def __init__(self, num_classes= 1000, training=True):
        super(InceptionV101, self).__init__()
        self.training = training
        self.conv = nn.Sequential(nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=3),
                      nn.ReLU(inplace=True),
                      nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
                      nn.Conv2d(in_channels=64, out_channels=64, kernel_size=1, stride=1),
                      nn.ReLU(inplace=True),
                      nn.Conv2d(in_channels=64, out_channels=192, kernel_size=3, stride=1, padding=1),
                      nn.ReLU(inplace=True),
                      nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        
        self.inception1 = inceptionv1_block(in_channels= 192, out_channels1= 64, out_channels2_step1= 96, out_channels2_step2=128, out_channels3_step1=16, out_channels3_step2=32, out_channels4=32)
        self.inception2 = inceptionv1_block(in_channels= 256, out_channels1= 128, out_channels2_step1= 128, out_channels2_step2=192, out_channels3_step1=32, out_channels3_step2=96, out_channels4=64)
        self.maxpooling1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.inception3 = inceptionv1_block(in_channels= 480, out_channels1= 192, out_channels2_step1= 96, out_channels2_step2=208, out_channels3_step1=16, out_channels3_step2=48, out_channels4=64)

        if self.training == True:
            self.auxiliary1 = auxiliary_classifiers(in_channels=512,out_channels=num_classes)

        self.inception4 = inceptionv1_block(in_channels=512 ,out_channels1=160, out_channels2_step1=112, out_channels2_step2=224, out_channels3_step1=24, out_channels3_step2=64, out_channels4=64)
        self.inception5 = inceptionv1_block(in_channels=512, out_channels1=128, out_channels2_step1=128, out_channels2_step2=256, out_channels3_step1=24, out_channels3_step2=64, out_channels4=64)
        self.inception6 = inceptionv1_block(in_channels=512, out_channels1=112, out_channels2_step1=144, out_channels2_step2=288, out_channels3_step1=32, out_channels3_step2=64, out_channels4=64)

        if self.training == True:
            self.auxiliary2 = auxiliary_classifiers(in_channels=528,out_channels=num_classes)

        self.inception7 = inceptionv1_block(in_channels=528, out_channels1=256, out_channels2_step1=160, out_channels2_step2=320, out_channels3_step1=32, out_channels3_step2=128, out_channels4=128)
        self.maxpooling2 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.inception8 = inceptionv1_block(in_channels=832, out_channels1=256, out_channels2_step1=160, out_channels2_step2=320, out_channels3_step1=32, out_channels3_step2=128, out_channels4=128)
        self.inception9 = inceptionv1_block(in_channels=832, out_channels1=384, out_channels2_step1=192, out_channels2_step2=384, out_channels3_step1=48, out_channels3_step2=128, out_channels4=128)

        self.avgpooling = nn.AvgPool2d(kernel_size=7,stride=1)
        self.dropout = nn.Dropout(p=0.4)
        self.fc = nn.Linear(in_features=1024,out_features=num_classes)

    def forward(self, x):
        x = self.conv(x)
        x = self.inception1(x)
        x = self.inception2(x)
        x = self.maxpooling1(x)
        x = self.inception3(x)
        aux1 = self.auxiliary1(x)
        x = self.inception4(x)
        x = self.inception5(x)
        x = self.inception6(x)
        aux2 = self.auxiliary2(x)
        x = self.inception7(x)
        x = self.maxpooling2(x)
        x = self.inception8(x)
        x = self.inception9(x)
        x = self.avgpooling(x)
        x = self.dropout(x)
        x = torch.flatten(x, start_dim=1)
        out = self.fc(x)

        if self.training == True:
            return aux1, aux2, out

        else:
            return out

In [6]:
if __name__=='__main__': 
    model = InceptionV101()
    #print(model)
    input = torch.randn(1, 3, 224, 224)
    aux1, aux2, out = model(input)
    print(aux1.shape)
    print(aux2.shape)
    print(out.shape)
    print(out)
    

torch.Size([1, 1000])
torch.Size([1, 1000])
torch.Size([1, 1000])
tensor([[ 2.8879e-02,  2.1250e-02,  8.5563e-03, -1.0167e-02, -1.1356e-03,
         -1.4748e-02, -1.6803e-02, -2.6719e-02,  8.0622e-03, -1.1865e-02,
          1.8017e-02,  2.1338e-02, -5.3641e-04, -5.2829e-03, -2.2371e-02,
         -1.1254e-02, -1.8064e-02, -3.8185e-03, -1.8432e-02,  1.4625e-02,
          3.6435e-03, -6.1388e-03, -6.4893e-03,  6.5762e-03,  1.0503e-02,
         -1.6460e-02,  1.5319e-02, -1.0891e-02, -4.6183e-03,  1.7152e-02,
          2.3439e-02, -2.3407e-03, -2.6918e-02, -4.0769e-02,  3.1234e-02,
          3.3199e-02, -1.7392e-02, -2.9165e-02, -3.0783e-02,  1.0317e-02,
          2.5190e-02, -1.1849e-02,  3.3384e-03,  1.1312e-02,  2.4642e-02,
         -4.0155e-02,  2.1663e-02, -2.6069e-02,  1.2608e-02,  1.4697e-02,
          3.1248e-03,  8.1907e-03, -3.0381e-02,  1.9432e-02, -2.5074e-02,
          1.9484e-02,  1.2974e-02,  2.5273e-02, -1.0886e-02, -3.3258e-02,
         -1.3697e-02,  2.7204e-02, -2.3012e-02

In [84]:
def ConvBNReLU(in_channels, out_channels, kernel_size):
    return nn.Sequential(
        nn.Conv2d(in_channels= in_channels, out_channels= out_channels, kernel_size= kernel_size, stride= 1, padding= kernel_size//2),
        nn.BatchNorm2d(out_channels),
        nn.ReLU6(inplace= True)
    )

In [85]:
class InceptionV1Module(nn.Module):
    def __init__(self, in_channels,out_channels1, out_channels2reduce,out_channels2, out_channels3reduce, out_channels3, out_channels4):
        super(InceptionV1Module, self).__init__()

        self.branch1_conv  = ConvBNReLU(in_channels= in_channels,         out_channels= out_channels1,       kernel_size= 1)

        self.branch2_conv1 = ConvBNReLU(in_channels= in_channels,         out_channels= out_channels2reduce, kernel_size= 1)
        self.branch2_conv2 = ConvBNReLU(in_channels= out_channels2reduce, out_channels= out_channels2,       kernel_size= 3)

        self.branch3_conv1 = ConvBNReLU(in_channels= in_channels,         out_channels= out_channels3reduce, kernel_size= 1)
        self.branch3_conv2 = ConvBNReLU(in_channels= out_channels3reduce, out_channels= out_channels3,       kernel_size= 5)

        self.branch4_pool = nn.MaxPool2d(kernel_size= 3,stride= 1,padding= 1)
        self.branch4_conv1 = ConvBNReLU(in_channels= in_channels,         out_channels= out_channels4,       kernel_size= 1)
        
    def forward(self,x):
        out1 = self.branch1_conv(x)
        out2 = self.branch2_conv2(self.branch2_conv1(x))
        out3 = self.branch3_conv2(self.branch3_conv1(x))
        out4 = self.branch4_conv1(self.branch4_pool(x))
        out = torch.cat([out1, out2, out3, out4], dim=1)
        return out

In [86]:
class InceptionAux(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(InceptionAux, self).__init__()

        self.auxiliary_avgpool= nn.AvgPool2d(kernel_size= 5, stride= 3)

        self.auxiliary_conv1= ConvBNReLU(in_channels= in_channels, out_channels= 128, kernel_size= 1)

        self.auxiliary_linear1= nn.Linear(in_features= 128 * 4 * 4, out_features= 1024)
        self.auxiliary_relu= nn.ReLU6(inplace= True)
        self.auxiliary_dropout= nn.Dropout(p= 0.7)
        
        self.auxiliary_linear2= nn.Linear(in_features= 1024, out_features= out_channels)
        
    def forward(self, x):
        x = self.auxiliary_conv1(self.auxiliary_avgpool(x))
        x = x.view(x.size(0), -1)
        x= self.auxiliary_relu(self.auxiliary_linear1(x))
        out = self.auxiliary_linear2(self.auxiliary_dropout(x))
        return out

In [93]:
class InceptionV1(nn.Module):
    def __init__(self, num_classes= 1000, Traing= True):
        super(InceptionV1, self).__init__()
        self.stage= Traing

        #sub-Module-01: A series of CNN & Pool
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels= 3,  out_channels= 64,  kernel_size= 7, stride= 2, padding= 3),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size= 3, stride= 2, padding= 1),
            nn.Conv2d(in_channels= 64, out_channels= 64,  kernel_size= 1, stride= 1),
            nn.BatchNorm2d(64),
            nn.Conv2d(in_channels= 64, out_channels= 192, kernel_size= 3, stride= 1, padding= 1),
            nn.BatchNorm2d(192),
            nn.MaxPool2d(kernel_size= 3, stride= 2, padding= 1),
        )
        
        #sub-Module-02: 3* InceptionV1  
        self.block2 = nn.Sequential(
            InceptionV1Module(in_channels= 192, out_channels1= 64,  out_channels2reduce= 96,  out_channels2= 128, out_channels3reduce= 16, out_channels3= 32, out_channels4= 32),
            InceptionV1Module(in_channels= 256, out_channels1= 128, out_channels2reduce= 128, out_channels2= 192, out_channels3reduce= 32, out_channels3= 96, out_channels4= 64),
            nn.MaxPool2d(kernel_size= 3, stride= 2, padding= 1),
            InceptionV1Module(in_channels= 480, out_channels1= 192, out_channels2reduce= 96,  out_channels2= 208, out_channels3reduce= 16, out_channels3= 48, out_channels4= 64)
        )
        
        #sub-Module-03: 3* InceptionV1 + 1* FC
        if self.stage == True:
            self.aux_logits1 = InceptionAux(in_channels=512,out_channels= num_classes)
        self.block3 = nn.Sequential(
            InceptionV1Module(in_channels= 512, out_channels1= 160, out_channels2reduce= 112, out_channels2= 224, out_channels3reduce= 24, out_channels3= 64, out_channels4= 64),
            InceptionV1Module(in_channels= 512, out_channels1= 128, out_channels2reduce= 128, out_channels2= 256, out_channels3reduce= 24, out_channels3= 64, out_channels4= 64),
            InceptionV1Module(in_channels= 512, out_channels1= 112, out_channels2reduce= 144, out_channels2= 288, out_channels3reduce= 32, out_channels3= 64, out_channels4= 64),
        )
   
        #sub-Module-04: 3* InceptionV1 + 2* FC
        if self.stage == True:
            self.aux_logits2 = InceptionAux(in_channels= 528, out_channels= num_classes)           
        self.block4 = nn.Sequential(
            InceptionV1Module(in_channels= 528, out_channels1= 256, out_channels2reduce= 160, out_channels2= 320, out_channels3reduce= 32, out_channels3= 128, out_channels4= 128),
            nn.MaxPool2d(kernel_size= 3, stride= 2, padding= 1),
            InceptionV1Module(in_channels= 832, out_channels1= 256, out_channels2reduce= 160, out_channels2= 320, out_channels3reduce= 32, out_channels3= 128, out_channels4= 128),
            InceptionV1Module(in_channels= 832, out_channels1= 384, out_channels2reduce= 192, out_channels2= 384, out_channels3reduce= 48, out_channels3= 128, out_channels4= 128),
        )
        self.block4_1 = nn.Sequential(
            nn.AvgPool2d(kernel_size= 7, stride= 1),
            nn.Dropout(p= 0.4),
            nn.Flatten(),
            nn.Linear(in_features= 1024, out_features= num_classes)
        )

        
    def forward(self, x):
        x = self.block1(x)
        aux1 = x = self.block2(x)
        aux2 = x = self.block3(x)
        x = self.block4(x)
        out = self.block4_1(x)
        if self.stage == True:
            aux1 = self.aux_logits1(aux1)
            aux2 = self.aux_logits2(aux2)
            return aux1, aux2, out
        else:
            return out

In [94]:
if __name__=='__main__':
    model = InceptionV1()
    #print(model)

    input = torch.randn(1, 3, 224, 224)
    aux1, aux2, out = model(input)
    print(aux1.shape)
    print(aux2.shape)
    print(out.shape)

torch.Size([1, 1000])
torch.Size([1, 1000])
torch.Size([1, 1000])
